# ASL video augmentation (30 clips per class)

Generates augmented copies of existing `.mp4` files under `Videos/ASL/<class>/` until each class folder contains **30** videos.

- **Skips** classes that already have ≥ 30 videos.
- **Appends** new files as the next integer names (`11.mp4`, `12.mp4`, …) after your current `0.mp4 …` files.
- **Does not use horizontal flip** (can swap dominant hand in ASL).

**pip note:** Lines starting with `ERROR: pip's dependency resolver…` are usually **warnings** about **other packages already installed** in this Python environment—not OpenCV failing to install. Here the conflict comes from **`numpy` 2.x** vs packages that still expect **`numpy` &lt; 2** (e.g. scipy, langchain, ai-hedge-fund). Keeping NumPy on **1.26–1.x** (below) avoids that on typical setups; for a completely clean slate, use a **virtual environment** only for this course project.

Install dependencies (run once):

```bash
pip install "opencv-python-headless>=4.8" "numpy>=1.26,<2"
```

In [3]:
# Optional: install if needed (numpy < 2 avoids clashes with scipy / langchain in shared envs)
%pip install -q "opencv-python-headless>=4.8" "numpy>=1.26,<2"


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from __future__ import annotations

import random
from pathlib import Path

import cv2
import numpy as np

# --- config ---
PROJECT_ROOT = Path(".").resolve()
ASL_ROOT = PROJECT_ROOT / "Videos" / "ASL"
TARGET_PER_CLASS = 30
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)

## Augmentation helpers

Combines mild **brightness/contrast**, **HSV jitter**, **Gaussian noise**, **small rotation**, and **temporal cropping** (drops a small random prefix/suffix of frames).

In [5]:
def _adjust_brightness_contrast(bgr: np.ndarray, alpha: float, beta: float) -> np.ndarray:
    return np.clip(bgr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


def _add_gaussian_noise(bgr: np.ndarray, sigma: float) -> np.ndarray:
    noise = RNG.normal(0.0, sigma, bgr.shape).astype(np.float32)
    return np.clip(bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def _hsv_jitter(bgr: np.ndarray, dh: float, ds: float, dv: float) -> np.ndarray:
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + dh) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * ds, 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * dv, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def _rotate_small(bgr: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w = bgr.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(
        bgr,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT101,
    )


def augment_frame(bgr: np.ndarray) -> np.ndarray:
    """Apply a random composition of spatial / appearance augmentations."""
    out = bgr

    if RNG.random() < 0.95:
        alpha = float(RNG.uniform(0.82, 1.18))
        beta = float(RNG.uniform(-28, 28))
        out = _adjust_brightness_contrast(out, alpha, beta)

    if RNG.random() < 0.6:
        dh = float(RNG.uniform(-4, 4))
        ds = float(RNG.uniform(0.88, 1.12))
        dv = float(RNG.uniform(0.90, 1.10))
        out = _hsv_jitter(out, dh, ds, dv)

    if RNG.random() < 0.55:
        sigma = float(RNG.uniform(2.0, 7.5))
        out = _add_gaussian_noise(out, sigma)

    if RNG.random() < 0.45:
        angle = float(RNG.uniform(-4.5, 4.5))
        out = _rotate_small(out, angle)

    return out


def temporal_crop(frames: list[np.ndarray]) -> list[np.ndarray]:
    n = len(frames)
    if n <= 4:
        return frames
    max_each = max(1, int(0.12 * n))
    drop_start = int(RNG.integers(0, max_each + 1))
    drop_end = int(RNG.integers(0, max_each + 1))
    if n - drop_start - drop_end < 3:
        return frames
    return frames[drop_start : n - drop_end]

In [6]:
def read_video(path: Path) -> tuple[list[np.ndarray], float]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-3:
        fps = 25.0
    frames: list[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames read: {path}")
    return frames, float(fps)


def write_video(path: Path, frames: list[np.ndarray], fps: float) -> None:
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(path), fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Cannot open VideoWriter for {path}")
    for f in frames:
        if f.shape[0] != h or f.shape[1] != w:
            f = cv2.resize(f, (w, h), interpolation=cv2.INTER_LINEAR)
        writer.write(f)
    writer.release()


def list_numeric_mp4s(folder: Path) -> list[Path]:
    paths = sorted(folder.glob("*.mp4"), key=lambda p: int(p.stem))
    return paths


def augment_one_video(src: Path, dst: Path) -> None:
    frames, fps = read_video(src)
    frames = temporal_crop(frames)
    aug_frames = [augment_frame(fr) for fr in frames]
    write_video(dst, aug_frames, fps)

## Frame size per clip (ASL)

Under **`Videos/ASL`**, for each `{class}/{video}.mp4`, OpenCV records **frame width**, **height**, **channels** from the first decoded frame, **`n_frames` by decoding to end-of-file**, approximate **pixels/bytes per frame**, and approximate **pixels in the clip** (`WxH × n_frames`). Also prints histograms over resolutions and channels.

In [12]:
def probe_video_frame_info(path: Path) -> tuple[int, int, int, int]:
    """Return (width_px, height_px, channels, n_frames).

    WxH taken from first decoded pixel buffer when possible (matches OpenCV).
    `n_frames` is an exact count via sequential reads to end-of-stream.
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    w_prop = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_prop = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames = 0
    ch = -1
    w, h = w_prop, h_prop
    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            break
        if n_frames == 0:
            h, w = frame.shape[:2]
            ch = frame.shape[2] if frame.ndim == 3 else 1
        n_frames += 1
    cap.release()
    return w, h, ch, n_frames


records: list[dict] = []

class_dirs_sorted = sorted(
    [p for p in ASL_ROOT.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower(),
)
assert ASL_ROOT.is_dir(), f"Missing folder: {ASL_ROOT}"

for cls_dir in class_dirs_sorted:
    for vid in list_numeric_mp4s(cls_dir):
        w, h, ch, n_frames = probe_video_frame_info(vid)
        bytes_per_px = float(w * h * ch) if ch > 0 else float("nan")
        records.append(
            {
                "class": cls_dir.name,
                "video": vid.stem,
                "width": w,
                "height": h,
                "channels": ch,
                "n_frames": n_frames,
                "WxH": f"{w}×{h}",
                "pixels_per_frame": w * h,
                "bytes_per_frame_approx": bytes_per_px,
                "pixels_total_approx": int(w * h * n_frames) if ch > 0 and n_frames else 0,
            }
        )

try:
    import pandas as pd

    df = pd.DataFrame.from_records(records)
    df["video"] = df["video"].astype(int)
    df = df.sort_values(["class", "video"]).reset_index(drop=True)
    display(df)
except ImportError:
    for row in sorted(records, key=lambda r: (r["class"].lower(), int(r["video"]))):
        print(
            f"{row['class']}/{row['video']}.mp4  WxH={row['WxH']}  "
            f"channels={row['channels']}  n_frames={row['n_frames']}  "
            f"~bytes/frame={row['bytes_per_frame_approx']}"
        )

from collections import Counter

res_hist = Counter((r["width"], r["height"]) for r in records)
print("\nDistinct frame sizes [width×height]", len(res_hist))
for (w, h), cnt in res_hist.most_common():
    print(f"  {w}×{h}: {cnt} clip(s)")

ch_hist = Counter(r["channels"] for r in records)
print("\nChannel counts (from first decoded frame):", dict(ch_hist))

_nf = sorted(r["n_frames"] for r in records)
print(
    "\nFrames per clip — min/med/max:",
    (_nf[0] if _nf else 0),
    (_nf[len(_nf) // 2] if _nf else 0),
    (_nf[-1] if _nf else 0),
)
print("\nTotal clips:", len(records))

,class,video,width,height,channels,n_frames,WxH,pixels_per_frame,bytes_per_frame_approx,pixels_total_approx
0,africa,0,480,320,3,44,480×320,153600,460800.0,6758400
1,africa,1,320,240,3,146,320×240,76800,230400.0,11212800
2,africa,2,1280,720,3,65,1280×720,921600,2764800.0,59904000
3,africa,3,1920,1080,3,48,1920×1080,2073600,6220800.0,99532800
4,africa,4,1920,1080,3,41,1920×1080,2073600,6220800.0,85017600
...,...,...,...,...,...,...,...,...,...,...
745,where,25,656,370,3,45,656×370,242720,728160.0,10922400
746,where,26,848,480,3,77,848×480,407040,1221120.0,31342080
747,where,27,480,320,3,52,480×320,153600,460800.0,7987200
748,where,28,320,240,3,95,320×240,76800,230400.0,7296000



Distinct frame sizes [width×height] 15
  1920×1080: 199 clip(s)
  480×320: 97 clip(s)
  640×480: 91 clip(s)
  320×240: 85 clip(s)
  1280×720: 78 clip(s)
  656×370: 60 clip(s)
  736×414: 31 clip(s)
  320×180: 27 clip(s)
  628×480: 23 clip(s)
  640×360: 15 clip(s)
  854×480: 14 clip(s)
  720×480: 13 clip(s)
  626×360: 8 clip(s)
  852×480: 6 clip(s)
  848×480: 3 clip(s)

Channel counts (from first decoded frame): {3: 750}

Frames per clip — min/med/max: 21 65 255

Total clips: 750


## Frame size per clip (SGSL)

Same logic as ASL using **`probe_video_frame_info`** (run the ASL probe cell above first). Processes each `{class}/{video}.mp4` under **`Videos/SGSL`**. This cell defines **`SGSL_ROOT`** from **`PROJECT_ROOT`** (or the notebook working directory), so it still works if you have not re-run the config cell after `SGSL_ROOT` was added there.

In [16]:
from pathlib import Path

_proj = globals().get("PROJECT_ROOT") or Path(".").resolve()
SGSL_ROOT = _proj / "Videos" / "SGSL"

records_sgsl: list[dict] = []

class_dirs_sgsl = sorted(
    [p for p in SGSL_ROOT.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower(),
)
assert SGSL_ROOT.is_dir(), f"Missing folder: {SGSL_ROOT}"

for cls_dir in class_dirs_sgsl:
    for vid in list_numeric_mp4s(cls_dir):
        w, h, ch, n_frames = probe_video_frame_info(vid)
        bytes_per_px = float(w * h * ch) if ch > 0 else float("nan")
        records_sgsl.append(
            {
                "class": cls_dir.name,
                "video": vid.stem,
                "width": w,
                "height": h,
                "channels": ch,
                "n_frames": n_frames,
                "WxH": f"{w}×{h}",
                "pixels_per_frame": w * h,
                "bytes_per_frame_approx": bytes_per_px,
                "pixels_total_approx": int(w * h * n_frames) if ch > 0 and n_frames else 0,
            }
        )

try:
    import pandas as pd

    df_sgsl = pd.DataFrame.from_records(records_sgsl)
    df_sgsl["video"] = df_sgsl["video"].astype(int)
    df_sgsl = df_sgsl.sort_values(["class", "video"]).reset_index(drop=True)
    display(df_sgsl)
except ImportError:
    for row in sorted(
        records_sgsl, key=lambda r: (r["class"].lower(), int(r["video"]))
    ):
        print(
            f"{row['class']}/{row['video']}.mp4  WxH={row['WxH']}  "
            f"channels={row['channels']}  n_frames={row['n_frames']}  "
            f"~bytes/frame={row['bytes_per_frame_approx']}"
        )

from collections import Counter

_pre = "[SGSL] "
res_hist_s = Counter((r["width"], r["height"]) for r in records_sgsl)
print(f"\n{_pre}Distinct frame sizes [width×height]", len(res_hist_s))
for (w, h), cnt in res_hist_s.most_common():
    print(f"  {w}×{h}: {cnt} clip(s)")

ch_hist_s = Counter(r["channels"] for r in records_sgsl)
print(f"\n{_pre}Channel counts (first frame):", dict(ch_hist_s))

_nf_s = sorted(r["n_frames"] for r in records_sgsl)
print(
    f"\n{_pre}Frames per clip — min/med/max:",
    (_nf_s[0] if _nf_s else 0),
    (_nf_s[len(_nf_s) // 2] if _nf_s else 0),
    (_nf_s[-1] if _nf_s else 0),
)
print(f"\n{_pre}Total clips:", len(records_sgsl))

,class,video,width,height,channels,n_frames,WxH,pixels_per_frame,bytes_per_frame_approx,pixels_total_approx
0,Bedok,0,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
1,Bedok,1,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
2,Bedok,2,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
3,Bedok,3,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
4,Bedok,4,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
...,...,...,...,...,...,...,...,...,...,...
175,Orchard,25,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
176,Orchard,26,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
177,Orchard,27,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000
178,Orchard,28,1920,1080,3,60,1920×1080,2073600,6220800.0,124416000



[SGSL] Distinct frame sizes [width×height] 1
  1920×1080: 180 clip(s)

[SGSL] Channel counts (first frame): {3: 180}

[SGSL] Frames per clip — min/med/max: 60 60 60

[SGSL] Total clips: 180


## Run augmentation for every ASL class

Cycles through existing clips in round-robin order so augmentations spread across all originals.

In [7]:
def fill_class_to_target(class_dir: Path, target: int) -> dict:
    """Return summary counts for one gloss folder."""
    mp4s = list_numeric_mp4s(class_dir)
    n = len(mp4s)
    summary = {"class": class_dir.name, "before": n, "generated": 0, "after": n}

    if n >= target:
        summary["after"] = n
        return summary

    if n == 0:
        raise RuntimeError(f"No source videos in {class_dir}")

    next_idx = max(int(p.stem) for p in mp4s) + 1
    sources = mp4s.copy()
    si = 0

    while n < target:
        src = sources[si % len(sources)]
        si += 1
        dst = class_dir / f"{next_idx}.mp4"
        augment_one_video(src, dst)
        next_idx += 1
        n += 1
        summary["generated"] += 1

    summary["after"] = n
    return summary


assert ASL_ROOT.is_dir(), f"Missing folder: {ASL_ROOT}"

class_dirs = sorted([p for p in ASL_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
rows: list[dict] = []

for d in class_dirs:
    rows.append(fill_class_to_target(d, TARGET_PER_CLASS))

for r in rows:
    print(
        f"{r['class']:<14} before={r['before']:>2}  +{r['generated']:<2}  -> after={r['after']}"
    )

total_gen = sum(r["generated"] for r in rows)
print("---")
print(f"Classes processed: {len(rows)}  |  Total new videos written: {total_gen}")

africa         before=11  +19  -> after=30
australia      before= 9  +21  -> after=30
basement       before= 8  +22  -> after=30
bathroom       before= 9  +21  -> after=30
behind         before= 8  +22  -> after=30
cafeteria      before= 7  +23  -> after=30
church         before= 5  +25  -> after=30
city           before=11  +19  -> after=30
college        before=11  +19  -> after=30
country        before= 7  +23  -> after=30
east           before= 9  +21  -> after=30
here           before= 8  +22  -> after=30
home           before= 8  +22  -> after=30
hospital       before= 6  +24  -> after=30
japan          before= 7  +23  -> after=30
north          before= 9  +21  -> after=30
office         before= 7  +23  -> after=30
party          before= 9  +21  -> after=30
restaurant     before= 9  +21  -> after=30
room           before= 9  +21  -> after=30
russia         before= 9  +21  -> after=30
school         before=11  +19  -> after=30
south          before=10  +20  -> after=30
west       

## Optional: preview one augmented clip

Uncomment and set `PREVIEW_CLASS` to a folder name under `ASL`.

In [8]:
# PREVIEW_CLASS = "church"
# d = ASL_ROOT / PREVIEW_CLASS
# originals = list_numeric_mp4s(d)
# if originals:
#     tmp = Path("_preview_aug.mp4")
#     augment_one_video(originals[0], tmp)
#     print("Wrote", tmp.resolve())